# Road Damage Object Detection — Model Comparison

Compares **YOLOv8s**, **YOLO11s** and **YOLO26s**, trained with identical settings in the three training notebooks.

Requirements covered: **mAP · Precision | Recall · IoU · Inference speed / FPS · Model size · Deployment efficiency**.

**Before running:** run all three training notebooks first. Each one writes `{model}_results.json` and its run folder to `road_damage/runs_v3/`.

- **Kaggle:** Add Input → *Your Work* → add the **output** of each of the three committed training notebooks; add the `ROBOFLOW_API_KEY` secret for section 8. Results are found automatically under `/kaggle/input`.
- **Colab:** results are read from `MyDrive/road_damage/runs_v3/`.

A GPU is optional here (only section 8 predicts on images).

## 1. Load results

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    BASE_DIR = "/kaggle/working/road_damage"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/road_damage"
PROJECT_DIR = f"{BASE_DIR}/runs_v3"
OUT_DIR = f"{BASE_DIR}/comparison"
os.makedirs(OUT_DIR, exist_ok=True)

MODELS = ["yolov8s", "yolo11s", "yolo26s"]
LABELS = {"yolov8s": "YOLOv8s", "yolo11s": "YOLO11s", "yolo26s": "YOLO26s"}

results, RUN_DIRS = {}, {}
for m in MODELS:
    # Colab: Drive folder. Kaggle: outputs of the training notebooks attached as inputs.
    hits = glob.glob(f"{PROJECT_DIR}/{m}_results.json") + \
           glob.glob(f"/kaggle/input/**/{m}_results.json", recursive=True)
    if hits:
        results[m] = json.load(open(hits[0]))
        RUN_DIRS[m] = os.path.join(os.path.dirname(hits[0]), m)
        print(f"{m:<8} <- {hits[0]}")
    else:
        print(f"Missing: {m}_results.json -> run (and on Kaggle, attach the output of) the {m} training notebook")
MODELS = [m for m in MODELS if m in results]
print("Loaded:", [LABELS[m] for m in MODELS])

## 2. Chart style
Fixed color per model (never by rank), validated for color-vision deficiency. Every chart also carries value labels, and section 3 is the table view.

In [ ]:
COLORS = {"yolov8s": "#2a78d6", "yolo11s": "#eb6834", "yolo26s": "#1baf7a"}
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#1f1f1e", "#5f5e5a", "#e4e3df"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "xtick.color": INK_2, "ytick.color": INK_2,
    "xtick.major.size": 0, "ytick.major.size": 0, "font.size": 11,
    "legend.frameon": False, "lines.linewidth": 2,
})

def grouped_bars(ax, categories, values_by_model, fmt="{:.3f}"):
    """values_by_model: {model: [value per category]}"""
    n = len(values_by_model)
    width = 0.8 / n
    x = np.arange(len(categories))
    for i, (m, vals) in enumerate(values_by_model.items()):
        pos = x - 0.4 + width * (i + 0.5)
        bars = ax.bar(pos, vals, width, color=COLORS[m], label=LABELS[m],
                      edgecolor=SURFACE, linewidth=2)
        for b, v in zip(bars, vals):
            ax.annotate(fmt.format(v), (b.get_x() + b.get_width() / 2, b.get_height()),
                        xytext=(0, 3), textcoords="offset points", ha="center",
                        fontsize=8.5, color=INK_2)
    ax.set_xticks(x, categories)
    ax.legend(loc="upper left", bbox_to_anchor=(0, 1.0), ncol=n)

def save(fig, name):
    fig.savefig(f"{OUT_DIR}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 3. Summary table

In [ ]:
COLUMNS = {
    "map50": "mAP50", "map50_95": "mAP50-95", "precision": "Precision", "recall": "Recall",
    "mean_iou": "Mean IoU", "gpu_fps": "GPU FPS (T4)", "cpu_fps": "CPU FPS",
    "gpu_ms": "GPU ms/img", "cpu_ms": "CPU ms/img", "size_mb": "best.pt (MB)",
    "params_m": "Params (M)", "gflops": "GFLOPs", "onnx_size_mb": "ONNX (MB)",
    "onnx_cpu_fps": "ONNX CPU FPS", "onnx_map50_95": "ONNX mAP50-95",
}
HIGHER_IS_BETTER = {"mAP50", "mAP50-95", "Precision", "Recall", "Mean IoU", "GPU FPS (T4)",
                    "CPU FPS", "ONNX CPU FPS", "ONNX mAP50-95"}

table = pd.DataFrame({LABELS[m]: {v: results[m][k] for k, v in COLUMNS.items()} for m in MODELS}).T
table.to_csv(f"{OUT_DIR}/comparison_table.csv")

def highlight_best(col):
    best = col.max() if col.name in HIGHER_IS_BETTER else col.min()
    return ["font-weight: bold; background-color: #e3eefb" if v == best else "" for v in col]

table.style.apply(highlight_best).format("{:.3f}", subset=list(table.columns[:5])) \
           .format("{:.1f}", subset=list(table.columns[5:]))

## 4. Accuracy: mAP, Precision, Recall, IoU

In [ ]:
metrics = ["map50", "map50_95", "precision", "recall", "mean_iou"]
fig, ax = plt.subplots(figsize=(11, 5))
grouped_bars(ax, [COLUMNS[k] for k in metrics], {m: [results[m][k] for k in metrics] for m in MODELS})
ax.set_ylim(0, 1.08)
ax.set_title("Accuracy on the test set")
save(fig, "accuracy")

In [ ]:
classes = list(results[MODELS[0]]["per_class"].keys())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
grouped_bars(axes[0], classes, {m: [results[m]["per_class"][c]["ap50_95"] for c in classes] for m in MODELS})
axes[0].set_title("AP50-95 per class")
axes[0].set_ylim(0, 1.08)
grouped_bars(axes[1], classes, {m: [results[m]["mean_iou_per_class"][c] for c in classes] for m in MODELS})
axes[1].set_title("Mean IoU per class")
axes[1].set_ylim(0, 1.08)
plt.tight_layout()
save(fig, "per_class")

## 5. Speed, size and deployment efficiency
Each measure has its own chart (different units, so no shared axis).

In [ ]:
speed = [("gpu_fps", "GPU FPS (T4, PyTorch)", "{:.0f}"),
         ("cpu_fps", "CPU FPS (PyTorch)", "{:.1f}"),
         ("onnx_cpu_fps", "CPU FPS (ONNX)", "{:.1f}")]
size = [("size_mb", "best.pt size (MB)", "{:.1f}"),
        ("params_m", "Parameters (M)", "{:.2f}"),
        ("gflops", "GFLOPs", "{:.1f}")]

for group, name in [(speed, "speed"), (size, "size")]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, (k, title, fmt) in zip(axes, group):
        vals = [results[m][k] for m in MODELS]
        bars = ax.bar([LABELS[m] for m in MODELS], vals, color=[COLORS[m] for m in MODELS],
                      edgecolor=SURFACE, linewidth=2, width=0.6)
        for b, v in zip(bars, vals):
            ax.annotate(fmt.format(v), (b.get_x() + b.get_width() / 2, b.get_height()),
                        xytext=(0, 3), textcoords="offset points", ha="center", fontsize=10, color=INK_2)
        ax.set_title(title)
        ax.set_ylim(0, max(vals) * 1.18)
    plt.tight_layout()
    save(fig, name)

In [ ]:
# Accuracy vs. deployment speed trade-off (top-right is best)
fig, ax = plt.subplots(figsize=(8, 5.5))
for m in MODELS:
    x, y = results[m]["onnx_cpu_fps"], results[m]["map50_95"]
    ax.scatter(x, y, s=60 + results[m]["size_mb"] * 8, color=COLORS[m],
               edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.annotate(f"{LABELS[m]}  ({results[m]['size_mb']:.0f} MB)", (x, y), xytext=(10, 6),
                textcoords="offset points", fontsize=10, color=INK)
ax.grid(True, axis="both")
ax.set_xlabel("ONNX CPU FPS (higher = faster deployment)")
ax.set_ylabel("mAP50-95 (test)")
ax.set_title("Accuracy vs. deployment speed (marker size = model size)")
xs = [results[m]["onnx_cpu_fps"] for m in MODELS]
ys = [results[m]["map50_95"] for m in MODELS]
ax.set_xlim(min(xs) * 0.85, max(xs) * 1.25)
ax.set_ylim(min(ys) - 0.03, max(ys) + 0.03)
save(fig, "tradeoff")

In [ ]:
# Does ONNX export keep accuracy?
pd.DataFrame({
    "PyTorch mAP50-95": {LABELS[m]: results[m]["map50_95"] for m in MODELS},
    "ONNX mAP50-95": {LABELS[m]: results[m]["onnx_map50_95"] for m in MODELS},
    "Drop": {LABELS[m]: results[m]["map50_95"] - results[m]["onnx_map50_95"] for m in MODELS},
    "ONNX speed-up on CPU": {LABELS[m]: results[m]["onnx_cpu_fps"] / results[m]["cpu_fps"] for m in MODELS},
}).round(4)

## 6. Training curves

In [ ]:
curves = {}
for m in MODELS:
    df = pd.read_csv(f"{RUN_DIRS[m]}/results.csv")
    df.columns = df.columns.str.strip()
    curves[m] = df

panels = [("metrics/mAP50-95(B)", "Validation mAP50-95"),
          ("metrics/mAP50(B)", "Validation mAP50"),
          ("val/box_loss", "Validation box loss"),
          ("train/box_loss", "Training box loss")]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (col, title) in zip(axes.flat, panels):
    for m in MODELS:
        if col in curves[m]:
            ax.plot(curves[m]["epoch"], curves[m][col], color=COLORS[m], label=LABELS[m])
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.grid(True, axis="both")
axes[0, 0].legend(loc="lower right")
plt.tight_layout()
save(fig, "training_curves")

for m in MODELS:
    df = curves[m]
    best_ep = int(df.loc[df["metrics/mAP50-95(B)"].idxmax(), "epoch"])
    print(f"{LABELS[m]:<8} trained {len(df)} epochs, best val mAP50-95 at epoch {best_ep}")

## 7. Overall ranking

Each measure is min-max normalised to 0–1 across the three models (1 = best), then weighted.
Weights reflect a deployment app: accuracy matters most, then CPU speed, then size. Change `WEIGHTS` to match your priorities and explain the choice in the report.

In [ ]:
WEIGHTS = {
    "map50_95": 0.40,
    "recall": 0.15,        # missing real damage is worse than a false alarm
    "mean_iou": 0.10,
    "onnx_cpu_fps": 0.20,
    "gpu_fps": 0.05,
    "size_mb": 0.10,       # lower is better
}
LOWER_IS_BETTER = {"size_mb"}

raw = pd.DataFrame({m: {k: results[m][k] for k in WEIGHTS} for m in MODELS}).T
norm = (raw - raw.min()) / (raw.max() - raw.min()).replace(0, 1)
for k in LOWER_IS_BETTER:
    norm[k] = 1 - norm[k]
norm["score"] = sum(norm[k] * w for k, w in WEIGHTS.items())
ranking = norm.sort_values("score", ascending=False)
ranking.index = [LABELS[m] for m in ranking.index]

BEST = ranking.index[0]
BEST_RUN = {v: k for k, v in LABELS.items()}[BEST]
print(f"Best overall: {BEST}")
ranking.round(3)

## 8. Side-by-side predictions on the same test images
Downloads the same Roboflow v3 test split used for training (needs the `ROBOFLOW_API_KEY` secret).

In [ ]:
!pip install -q -U ultralytics roboflow

In [ ]:
import glob, shutil, yaml
from roboflow import Roboflow

ROBOFLOW_VERSION = 3  # 70/20/10 split, Fit 640x640, 'unlabeled' null images excluded, no offline augmentation

# Roboflow API key comes from a secret named ROBOFLOW_API_KEY (never hard-code it)
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
else:
    from google.colab import userdata
    api_key = userdata.get("ROBOFLOW_API_KEY")

DATA_ROOT = "/tmp/road_damage"
shutil.rmtree(DATA_ROOT, ignore_errors=True)
rf = Roboflow(api_key=api_key)
dataset = rf.workspace("project-dvd12").project("road-damage-eul7g") \
            .version(ROBOFLOW_VERSION).download("yolov11", location=DATA_ROOT)

# Rewrite data.yaml with absolute paths so train/valid/test always resolve
DATA = f"{DATA_ROOT}/data.yaml"
with open(DATA) as f:
    data_cfg = yaml.safe_load(f)
data_cfg.update(path=DATA_ROOT, train="train/images", val="valid/images", test="test/images")
with open(DATA, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
NAMES = data_cfg["names"]

for split in ["train", "valid", "test"]:
    counts = {n: 0 for n in NAMES}
    for lbl in glob.glob(f"{DATA_ROOT}/{split}/labels/*.txt"):
        for line in open(lbl).read().split("\n"):
            if line.strip():
                counts[NAMES[int(float(line.split()[0]))]] += 1
    n_img = len(os.listdir(f"{DATA_ROOT}/{split}/images"))
    print(f"{split:<6} {n_img:>5} images | " + " | ".join(f"{k}: {v}" for k, v in counts.items()))
print(open(DATA).read())

In [ ]:
from ultralytics import YOLO

TEST_IMAGES = sorted(glob.glob(f"{DATA_ROOT}/test/images/*"))
models = {m: YOLO(f"{RUN_DIRS[m]}/weights/best.pt") for m in MODELS}
rng = np.random.default_rng(0)
sample = rng.choice(TEST_IMAGES, size=4, replace=False)

fig, axes = plt.subplots(len(sample), len(MODELS), figsize=(5 * len(MODELS), 5 * len(sample)))
for r, img in enumerate(sample):
    for c, m in enumerate(MODELS):
        pred = models[m].predict(img, imgsz=640, conf=0.35, verbose=False)[0]
        axes[r, c].imshow(pred.plot()[:, :, ::-1])
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(LABELS[m], color=COLORS[m], fontsize=15)
plt.tight_layout()
save(fig, "side_by_side")

## 9. Export the winner for the deployment app

In [ ]:
import shutil

DEPLOY_DIR = f"{BASE_DIR}/best_model"
os.makedirs(DEPLOY_DIR, exist_ok=True)
for ext in ["pt", "onnx"]:
    src = f"{RUN_DIRS[BEST_RUN]}/weights/best.{ext}"
    if os.path.exists(src):
        shutil.copy(src, f"{DEPLOY_DIR}/road_damage_{BEST_RUN}.{ext}")
print("Deployment weights:", os.listdir(DEPLOY_DIR))
print("Charts + table saved in:", OUT_DIR)

## 10. Discussion (fill in from the numbers above)

- **Accuracy:** which model has the highest mAP50-95 / mAP50? Is the gap meaningful (> ~0.01–0.02) or within noise on a 175-image test set?
- **Per class:** which class is hardest (likely *Crack*: thin, long, varied shapes)? Do all models struggle on the same class? Check the confusion matrices (Pothole ↔ Manhole confusion?).
- **Precision vs Recall:** for road inspection, a missed pothole (low recall) usually costs more than a false alarm.
- **IoU:** how tight are the boxes? Long diagonal cracks give loose axis-aligned boxes, so lower IoU there is expected.
- **Speed & size:** GPU FPS vs CPU FPS; is the model real-time (≥ 25–30 FPS) on the target hardware? How much does ONNX speed up CPU inference, and does it lose accuracy?
- **Training behaviour:** which model converged faster, and did any stop early (patience) or overfit (val loss rising)?
- **Decision:** which model goes to the deployment app and why (accuracy / speed / size trade-off)?
- **Next steps:** hyperparameter tuning on the best model (imgsz 800, lr0, optimizer, mosaic), more crack data, SAHI tiling for small damage.